In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import sys

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.shared.config import IMAGES_DIR, METRICS_PATH, PREDICTIONS_PATH, TEST_DF_PATH, RUN_MODE


In [ ]:
# Load và in metrics đã lưu (chạy script/shared/evaluate.py trước)
from src.shared.config import METRICS_PATH


if METRICS_PATH.exists():
    with METRICS_PATH.open(encoding="utf-8") as f:
        metrics = json.load(f)
    print(f"=== [{RUN_MODE}] {METRICS_PATH.name} ===")
    for metric, score in metrics.items():
        print(f"  {metric:>10}: {score * 100:.2f}")
else:
    print(f"Chưa có file metrics tại: {METRICS_PATH}")
    print("Hãy chạy: python script/shared/evaluate.py")


In [ ]:
# Tải lại dự đoán dưới dạng dictionary {imgid: caption}
with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
    preds = {int(p["imgid"]): p["caption"] for p in json.load(f)}

# Tải test dataset
test_df = pd.read_parquet(TEST_DF_PATH)

print(f"=== [{RUN_MODE}] {PREDICTIONS_PATH.name} ===")
print(f"Total predictions: {len(preds)}")
print()

# Lấy ngẫu nhiên 5 ảnh (đổi random_state để xem các ảnh khác)
sample_df = test_df.sample(5, random_state=42)

for _, row in sample_df.iterrows():
    imgid = int(row["imgid"])
    image_path = IMAGES_DIR / row["filepath"] / row["filename"]

    predicted_caption = preds.get(imgid, "NO PREDICTION FOUND")
    ground_truths = row["all_raws"]

    # Hiển thị ảnh
    plt.figure(figsize=(6, 4))
    try:
        img = Image.open(image_path).convert("RGB")
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        continue

    # In caption
    print(f"Image ID   : {imgid}")
    print(f"Prediction : {predicted_caption}")
    print(f"Ground Truths:")
    for i, gt in enumerate(ground_truths, 1):
        print(f"   {i}. {gt}")
    print("-" * 80 + "\n")
